# Feature Engineering

Dataset: IEEE-CIS Fraud Detection

| Step | Note |
|------|------|
| Memory Optimization | Downcasts data types to reduce memory usage |
| Data Loading | Merges transaction and identity data, sorts by time |
| Baseline Cleaning | Maps email domains, adds time features (hour/day), label encodes categorical variables |
| Domain-Specific Features | Adds aggregated statistics (mean/std transaction amounts per card), spending deviations, transaction velocity, and device-card aggregations |
| Output | Saves baseline and engineered datasets to Parquet files |

In [1]:
import pandas as pd
import numpy as np
import gc
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

DATA_DIR = 'data_raw/IEEE-CIS/'

# ============================================================
# 1. MEMORY OPTIMIZATION
# ============================================================
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Mem decreased to {end_mem:.2f} Mb ({100 * (start_mem - end_mem) / start_mem:.1f}% reduction)')
    return df

# ============================================================
# 2. LOAD DATA
# ============================================================
print("Loading data...")
train_trans = pd.read_csv(f'{DATA_DIR}/train_transaction.csv')
train_id    = pd.read_csv(f'{DATA_DIR}/train_identity.csv')
df = pd.merge(train_trans, train_id, on='TransactionID', how='left')

# Sort chronologically — required for temporal splitting
df = df.sort_values('TransactionDT').reset_index(drop=True)

# ============================================================
# 3. CHRONOLOGICAL TRAIN / VAL / TEST SPLIT  ← must happen FIRST
#    Everything below uses only train statistics to avoid
#    any future data leaking into the training features.
# ============================================================
n = len(df)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

df_train = df.iloc[:train_end].copy()
df_val   = df.iloc[train_end:val_end].copy()
df_test  = df.iloc[val_end:].copy()
print(f"Split: {len(df_train):,} train / {len(df_val):,} val / {len(df_test):,} test rows")
del df; gc.collect()

# ============================================================
# 4. BASELINE CLEANING  (fit on train, apply to both)
# ============================================================
print("Applying baseline cleaning...")

# -- Card combination composite key — more precise user proxy than card1 alone --
for df_ in [df_train, df_val, df_test]:
    df_['CardCombination'] = (
        df_['card1'].astype(str) + '_' +
        df_['card2'].astype(str) + '_' +
        df_['card3'].astype(str) + '_' +
        df_['card5'].astype(str)
    )

# -- Email domain mapping --
emails = {
    'gmail.com': 'google', 'att.net': 'at&t', 'twc.com': 'spectrum',
    'sc.rr.com': 'spectrum', 'nycap.rr.com': 'spectrum', 'charter.net': 'spectrum',
    'prodigy.net.mx': 'at&t', 'protonmail.com': 'protonmail', 'ptd.net': 'penn_telecom',
    'aim.com': 'aol', 'web.de': 'other', 'icloud.com': 'apple', 'hotmail.com': 'microsoft'
}
for c in ['P_emaildomain', 'R_emaildomain']:
    df_train[c] = df_train[c].map(emails).fillna('other')
    df_val[c]   = df_val[c].map(emails).fillna('other')
    df_test[c]  = df_test[c].map(emails).fillna('other')

# -- Time features (deterministic, no leakage) --
for df_ in [df_train, df_val, df_test]:
    df_['Transaction_hour'] = np.floor(df_['TransactionDT'] / 3600) % 24
    df_['Transaction_day']  = np.floor(df_['TransactionDT'] / (3600 * 24)) % 7

# -- Label Encoding: fit on train, apply to both --
# We collect all object columns, fit an encoder on train (using train+val
# categories to avoid unseen-label errors on val), then transform both.
cat_cols = [col for col in df_train.columns if pd.api.types.is_string_dtype(df_train[col])]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([df_train[col], df_val[col], df_test[col]],
                          ignore_index=True).astype(str)
    le.fit(combined)
    df_train[col] = le.transform(df_train[col].astype(str))
    df_val[col]   = le.transform(df_val[col].astype(str))
    df_test[col]  = le.transform(df_test[col].astype(str))
    encoders[col] = le

# Save baseline parquets (before domain-specific features)
print("Saving baseline parquets...")
# in section 4 (baseline save), add df_test:
df_train_base = reduce_mem_usage(df_train.copy())
df_val_base   = reduce_mem_usage(df_val.copy())
df_test_base  = reduce_mem_usage(df_test.copy())
df_train_base.to_parquet('data/iceee_baseline_train.parquet')
df_val_base.to_parquet('data/iceee_baseline_val.parquet')
df_test_base.to_parquet('data/iceee_baseline_test.parquet')
del df_train_base, df_val_base, df_test_base; gc.collect()

# ============================================================
# 5. DOMAIN-SPECIFIC FEATURE ENGINEERING
#    All aggregations computed on TRAIN only, then merged onto
#    test & val 
# ============================================================
print("Adding Domain-Specific Features...")

# Helper: compute group aggregation on train, merge onto other splits
def add_group_agg(df_tr, df_vl, df_te, group_col, value_col, agg_funcs):
    """
    Computes aggregation statistics from df_tr only, then left-merges
    the result onto df_tr, df_vl and df_te. Returns updated dataframes.
    """
    agg = df_tr.groupby(group_col)[value_col].agg(agg_funcs)
    agg.columns = [f'{group_col}_{value_col}_{fn}' for fn in agg_funcs]
    agg = agg.reset_index()
    df_tr = df_tr.merge(agg, on=group_col, how='left')
    df_vl = df_vl.merge(agg, on=group_col, how='left')
    df_te = df_te.merge(agg, on=group_col, how='left')
    return df_tr, df_vl, df_te

# A. AGGREGATED STATISTICS — mean & std transaction amount per card1
df_train, df_val, df_test = add_group_agg(df_train, df_val, df_test, 'card1', 'TransactionAmt', ['mean', 'std'])

# B. SPENDING DEVIATION
for df_ in [df_train, df_val, df_test]:
    df_['Amt_to_mean_card1'] = df_['TransactionAmt'] / df_['card1_TransactionAmt_mean']

# C. VELOCITY
count_map = df_train.groupby('card1')['TransactionID'].count().rename('card1_count')
df_train = df_train.merge(count_map.reset_index(), on='card1', how='left')
df_val   = df_val.merge(count_map.reset_index(), on='card1', how='left')
df_test  = df_test.merge(count_map.reset_index(), on='card1', how='left')

# D. DEVICE DIVERSITY
device_map = df_train.groupby('card1')['DeviceInfo'].nunique().rename('card1_Device_nuniq')
df_train = df_train.merge(device_map.reset_index(), on='card1', how='left')
df_val   = df_val.merge(device_map.reset_index(), on='card1', how='left')
df_test  = df_test.merge(device_map.reset_index(), on='card1', how='left')

# E. ROLLING FRAUD RATE per CardCombination and P_emaildomain
global_fraud_mean = df_train['isFraud'].mean()
for col in ['CardCombination', 'P_emaildomain']:
    fraud_rate_map = df_train.groupby(col)['isFraud'].mean()
    df_train[f'{col}_fraud_rate'] = df_train[col].map(fraud_rate_map).fillna(global_fraud_mean)
    df_val[f'{col}_fraud_rate']   = df_val[col].map(fraud_rate_map).fillna(global_fraud_mean)
    df_test[f'{col}_fraud_rate']  = df_test[col].map(fraud_rate_map).fillna(global_fraud_mean)

# F. TARGET ENCODING — k-fold on train, apply to val and test
from sklearn.model_selection import KFold

def kfold_target_encode(df_tr, df_vl, df_te, col, target='isFraud', n_splits=5):
    enc_col = f'{col}_target_enc'
    global_mean = df_tr[target].mean()
    df_tr[enc_col] = np.nan

    kf = KFold(n_splits=n_splits, shuffle=False)
    for train_idx, oof_idx in kf.split(df_tr):
        fold_map = df_tr.iloc[train_idx].groupby(col)[target].mean()
        df_tr.iloc[oof_idx, df_tr.columns.get_loc(enc_col)] = \
            df_tr.iloc[oof_idx][col].map(fold_map).fillna(global_mean).values

    full_map = df_tr.groupby(col)[target].mean()
    df_vl[enc_col] = df_vl[col].map(full_map).fillna(global_mean)
    df_te[enc_col] = df_te[col].map(full_map).fillna(global_mean)
    return df_tr, df_vl, df_te

for col in ['CardCombination', 'P_emaildomain', 'R_emaildomain']:
    df_train, df_val, df_test = kfold_target_encode(df_train, df_val, df_test, col)

# G. LOG TRANSFORM of TransactionAmt
for df_ in [df_train, df_val, df_test]:
    df_['TransactionAmt_log'] = np.log1p(df_['TransactionAmt'])

# Fill NaNs (rows for unseen values will be NaN → -999)
df_train = df_train.fillna(-999)
df_val   = df_val.fillna(-999)
df_test  = df_test.fillna(-999)

# Ensure all string-typed columns are int (label-encoded); catches both
# regular object and Arrow-backed StringDtype that dtype=='object' misses.
for col in df_train.columns:
    if pd.api.types.is_string_dtype(df_train[col]):
        df_train[col] = df_train[col].astype(np.int64)
for col in df_val.columns:
    if pd.api.types.is_string_dtype(df_val[col]):
        df_val[col] = df_val[col].astype(np.int64)
for col in df_test.columns:
    if pd.api.types.is_string_dtype(df_test[col]):
        df_test[col] = df_test[col].astype(np.int64)


# ============================================================
# 6. SAVE ENGINEERED DATASETS
# ============================================================
print("Saving engineered parquets...")

df_train_eng = reduce_mem_usage(df_train)
df_val_eng   = reduce_mem_usage(df_val)
df_test_eng  = reduce_mem_usage(df_test)

df_train_eng.to_parquet('data/iceee_feature_train.parquet')
df_val_eng.to_parquet('data/iceee_feature_val.parquet')
df_test_eng.to_parquet('data/iceee_feature_test.parquet')

print()
print("Process Complete. Produced:")
print("  data/iceee_baseline_train.parquet")
print("  data/iceee_baseline_val.parquet")
print("  data/iceee_baseline_test.parquet")
print("  data/iceee_feature_train.parquet")
print("  data/iceee_feature_val.parquet")
print("  data/iceee_feature_test.parquet")

Loading data...
Split: 413,378 train / 88,581 val / 88,581 test rows
Applying baseline cleaning...
Saving baseline parquets...
Mem decreased to 650.87 Mb (52.8% reduction)
Mem decreased to 139.47 Mb (52.8% reduction)
Mem decreased to 139.47 Mb (52.8% reduction)
Adding Domain-Specific Features...
Saving engineered parquets...
Mem decreased to 666.64 Mb (52.8% reduction)
Mem decreased to 143.19 Mb (52.7% reduction)
Mem decreased to 143.19 Mb (52.7% reduction)

Process Complete. Produced:
  data/iceee_baseline_train.parquet
  data/iceee_baseline_val.parquet
  data/iceee_baseline_test.parquet
  data/iceee_feature_train.parquet
  data/iceee_feature_val.parquet
  data/iceee_feature_test.parquet
